In [1]:
import requests, pandas as pd
from datetime import datetime

VILLES = [
    {"nom": "Paris",     "lat": 48.8566, "lon": 2.3522},
    {"nom": "Lyon",      "lat": 45.7640, "lon": 4.8357},
    {"nom": "Marseille", "lat": 43.2965, "lon": 5.3698},
    {"nom": "Bordeaux",  "lat": 44.8378, "lon": -0.5792},
    {"nom": "Lille",     "lat": 50.6292, "lon": 3.0573},
]

def collecter(lat, lon):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    r = requests.get(url, params={
        "latitude": lat, "longitude": lon,
        "hourly": "pm10,pm2_5,nitrogen_dioxide,ozone",
        "timezone": "Europe/Paris", "past_days": 1
    }, timeout=10)
    return r.json()

rows = []
for v in VILLES:
    print(f"🔄 Collecte {v['nom']}...")
    data = collecter(v["lat"], v["lon"])
    for i, h in enumerate(data["hourly"]["time"]):
        rows.append({
            "ville": v["nom"], "datetime": h,
            "pm10": data["hourly"]["pm10"][i],
            "pm2_5": data["hourly"]["pm2_5"][i],
            "nitrogen_dioxide": data["hourly"]["nitrogen_dioxide"][i],
            "ozone": data["hourly"]["ozone"][i],
            "collecte_le": datetime.now().strftime("%Y-%m-%d %H:%M")
        })

df = pd.DataFrame(rows)
print(f"\n✅ {len(df)} lignes collectées pour {df['ville'].nunique()} villes !")
df.head()

🔄 Collecte Paris...
🔄 Collecte Lyon...
🔄 Collecte Marseille...
🔄 Collecte Bordeaux...
🔄 Collecte Lille...

✅ 720 lignes collectées pour 5 villes !


,ville,datetime,pm10,pm2_5,nitrogen_dioxide,ozone,collecte_le
0,Paris,2026-04-13T00:00,24.8,13.0,50.4,24.0,2026-04-14 21:02
1,Paris,2026-04-13T01:00,24.7,13.7,59.7,19.0,2026-04-14 21:02
2,Paris,2026-04-13T02:00,22.7,14.9,43.6,20.0,2026-04-14 21:02
3,Paris,2026-04-13T03:00,23.6,15.5,45.2,17.0,2026-04-14 21:02
4,Paris,2026-04-13T04:00,25.4,16.5,44.2,14.0,2026-04-14 21:02


In [3]:
# C2 - Nettoyage des données

df["datetime"] = pd.to_datetime(df["datetime"])

def indice(pm25):
    if pd.isna(pm25): return "Inconnu"
    elif pm25 <= 10:  return "Bon"
    elif pm25 <= 25:  return "Moyen"
    elif pm25 <= 50:  return "Mauvais"
    else:             return "Très mauvais"

df["indice_qualite"] = df["pm2_5"].apply(indice)
df = df.drop_duplicates()
for col in ["pm10","pm2_5","nitrogen_dioxide","ozone"]:
    df[col] = df[col].round(2)

print("✅ C2 - Nettoyage terminé !")
print(f"Lignes : {len(df)}")
print(f"Nulls  : {df.isnull().sum().sum()}")
print(f"\nRépartition qualité :")
print(df["indice_qualite"].value_counts())

✅ C2 - Nettoyage terminé !
Lignes : 720
Nulls  : 0

Répartition qualité :
indice_qualite
Bon        432
Moyen      283
Mauvais      5
Name: count, dtype: int64


In [4]:
# C3 - Agrégations des données

agregation_ville = df.groupby("ville").agg(
    pm10_moyen        = ("pm10",             "mean"),
    pm25_moyen        = ("pm2_5",            "mean"),
    no2_moyen         = ("nitrogen_dioxide", "mean"),
    ozone_moyen       = ("ozone",            "mean"),
    nb_mesures        = ("pm10",             "count")
).round(2).reset_index()

df["heure"] = df["datetime"].dt.hour
agregation_heure = df.groupby("heure").agg(
    pm25_moyen = ("pm2_5",            "mean"),
    no2_moyen  = ("nitrogen_dioxide", "mean")
).round(2).reset_index()

ville_max = agregation_ville.loc[agregation_ville["pm25_moyen"].idxmax()]
ville_min = agregation_ville.loc[agregation_ville["pm25_moyen"].idxmin()]

print("✅ C3 - Agrégations terminées !")
print(f"\n📊 Moyenne par ville :\n{agregation_ville}")
print(f"\n⏰ Top 5 heures polluées :\n{agregation_heure.sort_values('pm25_moyen', ascending=False).head()}")
print(f"\n🏭 Plus polluée : {ville_max['ville']} ({ville_max['pm25_moyen']} µg/m³)")
print(f"🌿 Plus saine   : {ville_min['ville']} ({ville_min['pm25_moyen']} µg/m³)")

df.to_csv("../data/air_quality_clean.csv", index=False)
agregation_ville.to_csv("../data/agregation_villes.csv", index=False)
print("\n💾 CSV sauvegardés !")

✅ C3 - Agrégations terminées !

📊 Moyenne par ville :
       ville  pm10_moyen  pm25_moyen  no2_moyen  ozone_moyen  nb_mesures
0   Bordeaux       10.34        6.07       7.88        65.31         144
1      Lille       16.67       12.05      14.45        57.75         144
2       Lyon       12.37        9.57      12.28        67.53         144
3  Marseille       14.88        9.64      10.85        74.33         144
4      Paris       15.68       11.23      30.08        49.62         144

⏰ Top 5 heures polluées :
    heure  pm25_moyen  no2_moyen
9       9       12.58      24.54
8       8       12.55      26.14
10     10       12.19      19.47
7       7       11.58      20.56
11     11       11.55      14.88

🏭 Plus polluée : Lille (12.05 µg/m³)
🌿 Plus saine   : Bordeaux (6.07 µg/m³)

💾 CSV sauvegardés !


In [5]:
# C4 - Création base de données PostgreSQL

from sqlalchemy import create_engine, Column, Integer, Float, String, DateTime, text
from sqlalchemy.orm import declarative_base, Session
from dotenv import load_dotenv
import os

load_dotenv("../.env")

DATABASE_URL = (
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

engine = create_engine(DATABASE_URL)
Base   = declarative_base()

class Ville(Base):
    __tablename__ = "villes"
    id        = Column(Integer, primary_key=True, autoincrement=True)
    nom       = Column(String(100), unique=True, nullable=False)
    latitude  = Column(Float, nullable=False)
    longitude = Column(Float, nullable=False)

class Mesure(Base):
    __tablename__ = "mesures"
    id               = Column(Integer, primary_key=True, autoincrement=True)
    ville            = Column(String(100), nullable=False)
    datetime         = Column(DateTime, nullable=False)
    pm10             = Column(Float)
    pm2_5            = Column(Float)
    nitrogen_dioxide = Column(Float)
    ozone            = Column(Float)
    indice_qualite   = Column(String(20))
    collecte_le      = Column(String(30))

class Agregation(Base):
    __tablename__ = "agregations"
    id          = Column(Integer, primary_key=True, autoincrement=True)
    ville       = Column(String(100), nullable=False)
    pm10_moyen  = Column(Float)
    pm25_moyen  = Column(Float)
    no2_moyen   = Column(Float)
    ozone_moyen = Column(Float)
    nb_mesures  = Column(Integer)

Base.metadata.create_all(engine)
print("✅ Tables créées : villes, mesures, agregations")

with Session(engine) as session:
    session.execute(text("TRUNCATE TABLE mesures, agregations, villes RESTART IDENTITY CASCADE"))
    session.commit()

    for v in VILLES:
        session.add(Ville(nom=v["nom"], latitude=v["lat"], longitude=v["lon"]))
    session.commit()
    print("✅ Villes insérées")

    for _, row in df.iterrows():
        session.add(Mesure(
            ville            = row["ville"],
            datetime         = row["datetime"],
            pm10             = row["pm10"],
            pm2_5            = row["pm2_5"],
            nitrogen_dioxide = row["nitrogen_dioxide"],
            ozone            = row["ozone"],
            indice_qualite   = row["indice_qualite"],
            collecte_le      = row["collecte_le"]
        ))
    session.commit()
    print(f"✅ {len(df)} mesures insérées")

    for _, row in agregation_ville.iterrows():
        session.add(Agregation(
            ville       = row["ville"],
            pm10_moyen  = row["pm10_moyen"],
            pm25_moyen  = row["pm25_moyen"],
            no2_moyen   = row["no2_moyen"],
            ozone_moyen = row["ozone_moyen"],
            nb_mesures  = int(row["nb_mesures"])
        ))
    session.commit()
    print("✅ Agrégations insérées")

print("\n🎉 Base de données prête !")

✅ Tables créées : villes, mesures, agregations
✅ Villes insérées
✅ 720 mesures insérées
✅ Agrégations insérées

🎉 Base de données prête !
